# AEGIS AI — Director Reasoning (Qwen3-VL) 🧠  [OPTIONAL / ADVANCED]

> ⚠️ **Ye notebook OPTIONAL hai — pehle wale 2 notebooks zaroori hain, ye nahi.**
> Ye un logon ke liye hai jo AEGIS ke **director** (kaunsa camera dikhao, kab replay chalao — ye faisle lene wala AI) ko aur smart banana chahte hain.

**Ye notebook 2 cheezein karta hai:**
1. **Inference setup** — Qwen3-VL-8B (Alibaba ka vision-language model) load karke ek **sample broadcast frame** par chalata hai aur match ka haal batata hai.
2. **LoRA fine-tuning ka outline** — agar aap is model ko BGMI broadcast style par fine-tune karna chaho to kaise karoge (steps + code skeleton).

**Steps:**
1. Settings → Accelerator → **GPU T4** on karo.
2. **Run** dabao. 4-bit quantization me model load hoga (~5-6 GB).
3. Cell 5 me sample frame ka intezam karo (neeche tarika likha hai).


In [ ]:
# Cell 1 — Latest transformers (Qwen3-VL support ke liye) + helpers install karo
!pip install -q git+https://github.com/huggingface/transformers accelerate bitsandbytes
print("Install complete ✔  (agar import error aaye to Kernel → Restart karke dobara chalao)")


In [ ]:
# Cell 2 — GPU aur VRAM check
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
    print("Note: 8B model bina quantization ke ~16GB leta — T4 (16GB) par 4-bit quantization ZAROORI hai.")
else:
    print("⚠️ GPU NAHI mila! Settings → Accelerator → GPU T4 select karo.")


## Step 1 — Model load karo (4-bit me) 📥

`Qwen/Qwen3-VL-8B-Instruct` HuggingFace se download hoga (~pehli baar 2-4 min). **4-bit quantization** isliye taaki 16GB wale T4 par aaram se chale.


In [ ]:
# Cell 3 — Qwen3-VL-8B load (4-bit quantized)
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"  # official Alibaba checkpoint, HuggingFace par free

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
print("Model download/load ho raha hai... (pehli baar thoda time lagega)")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("✔ Model taiyaar hai")


## Step 2 — Sample broadcast frame par test karo 🖼️

Notebook pehle `/kaggle/input/` me koi image dhoondhega (agar aapne dataset attach kiya hai to usi me se ek frame mil jayegi). Na mile to:
- Kaggle notebook me right side **+ Add data** se dataset jodho, ya
- Notebook ke *File* menu se ek BGMI broadcast ka screenshot **upload** karo `/kaggle/input/` me.

Phir model se director-style sawal puchenge: *"Match ka haal batao — kaunsi teams zinda hain, zone kahan hai, agla fight kahan hoga?"*


In [ ]:
# Cell 4 — Sample frame dhoondho
import glob

cands = []
for ext in ("jpg", "jpeg", "png", "webp"):
    cands += glob.glob(f"/kaggle/input/**/**/*.{ext}", recursive=True)
    cands += glob.glob(f"/kaggle/input/**/*.{ext}", recursive=True)
cands = sorted(set(cands))
FRAME = cands[0] if cands else None
if FRAME:
    print("Sample frame mili:", FRAME)
    print(f"(kul {len(cands)} images mili /kaggle/input me)")
else:
    print("⚠️ Koi sample image nahi mili.")
    print("Dataset attach karo (+ Add data) ya ek broadcast frame upload karo, phir ye cell dobara chalao.")


In [ ]:
# Cell 5 — Director-style reasoning: frame dekho, match ka haal batao
if FRAME is None:
    print("Pehle Cell 4 me frame ka intezam karo, phir ye cell chalao.")
else:
    from PIL import Image
    image = Image.open(FRAME).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": (
            "You are the AI director of a BGMI esports broadcast. "
            "Look at this broadcast frame and answer in simple Hinglish (Roman script): "
            "1) Kaunsi teams/players dikh rahe hain? "
            "2) Zone aur match ka haal kya hai? "
            "3) Agla important moment kahan ho sakta hai — camera wahan kyun dikhana chahiye?"
        )},
    ]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(model.device)
    print("Soch raha hai... ⏳")
    out = model.generate(**inputs, max_new_tokens=256)
    answer = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print("\n----- Director ka jawab -----\n")
    print(answer)


## Step 3 — LoRA/QLoRA fine-tuning OUTLINE 🛠️ (advanced, optional)

Agar aap chahte ho ki model **BGMI broadcast ki bhasha/style** me jawab de (jaise casters bolte hain), to use apne frames + captions par fine-tune kar sakte ho. Neeche poora outline hai — **imaandaar VRAM notes ke saath:**

**VRAM ki sachchai (16GB Tesla T4 par):**
- **Inference (upar jo kiya):** 4-bit me aaram se chalta hai ✔
- **QLoRA fine-tune (4-bit base + chhote LoRA adapters):** T4 par **ho sakta hai** lekin slow hoga — batch size 1-2, gradient accumulation 8-16 rakho. Agar `CUDA out of memory` aaye to batch aur chhota karo ya sequence length ghatao.
- **Full fine-tune (bina quantization, saare weights update):** Iske liye **40GB+ GPU (A100)** chahiye — Kaggle ke free T4 par **mat try karo**, paisa/time waste hoga.

**Fine-tune ke steps:**
1. Apna dataset banao: `(broadcast frame, director-style caption)` ke jodhe — JSONL file, har line: `{"image": "frame001.jpg", "caption": "..."}`.
2. `peft` + `trl` install karo.
3. Neeche wale code skeleton me `RUN_LORA = True` karo aur apne dataset ka path likho.
4. Train karo → sirf **LoRA adapters** save honge (chhoti file, ~100-200MB) → inhe base model ke upar load karke inference karo.

**Kitna data chahiye?** Koi jaadui number nahi — jitne zyada alag-alag tarah ke frames + achhe captions, utna behtar style seekhega. Chhote style-adaptation ke liye aam taur par kuch sau se kuch hazaar examples se shuruaat hoti hai.


In [ ]:
# Cell 6 — QLoRA fine-tuning SKELETON (RUN_LORA=True karo tabhi chalega)
RUN_LORA = False  # ← advanced users: True karo jab upar ka outline samajh aa jaye
TRAIN_JSONL = "/kaggle/input/<DATASET_SLUG>/director_captions.jsonl"  # apna path likho

if not RUN_LORA:
    print("RUN_LORA=False hai, isliye kuch nahi chala.")
    print("Jab fine-tune karna ho to RUN_LORA=True karo aur TRAIN_JSONL me apne captions ka path likho.")
else:
    # !pip install -q peft trl datasets   (pehle ye line uncomment karke chalao)
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig

    model = prepare_model_for_kbit_training(model)  # 4-bit base ko train-ready banao
    lora_cfg = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.1,
        # vision + language dono hisso ke attention/feedforward layers:
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()  # sirf ~1-2% params train honge — isi ko QLoRA kehte hain

    sft_cfg = SFTConfig(
        output_dir="/kaggle/working/qwen3vl_lora",
        num_train_epochs=3,
        per_device_train_batch_size=1,   # T4 par 1 se zyada mat rakho
        gradient_accumulation_steps=8,
        learning_rate=5e-5,
        fp16=True,
        logging_steps=10,
        save_steps=100,
    )
    print("Ab apna image+caption dataset load karke SFTTrainer(...) banao aur trainer.train() chalao.")
    print("Poora data-format example docs/KAGGLE_TRAINING.md ke Qwen3-VL section me milega.")


## ✅ Summary

- **Inference** (Cell 1-5): Qwen3-VL kisi bhi broadcast frame ko dekhkar director-style analysis de sakta hai — AEGIS ke director agent me yehi istemal hoga.
- **Fine-tuning** (Cell 6 outline): Optional hai. Pehle inference se kaam chalao; jab lage ki model ko BGMI style sikhana hai, tab QLoRA try karo.
- Yaad rakho: ye model **broadcast samajhne** ke liye hai, kill-feed detect karne ke liye nahi — uske liye pehle wale 2 notebooks hain.
